# YOLOv5 Inference
This Jupyter notebook shows how to run the trained YOLOv5 model on 2000x1600 13-bit color-depth microscope images for cell detection. Note that the model is trained on CellPose cell detections and not properly annotated images. So the detections at best can be as accurate as CellPose detections. Once the images are annotated properly, YOLOv5 should be trained again for better accuracy. 

The model requires two files:
- The ONNX weights file (.onnx file)
- The list of labels (.names file)

YOLOv5 is impleneted and trained in PyTorch framework. To run inference efficiently and without the need for PyTorch, the model is then converted to ONNX and is run using OpenCV dnn module below. 

To be able to run the converted model on GPU, OpenCV built with CUDA should be installed. Otherwise, the model runs on CPU with longer runtimes. 

### Required libraries

In [ ]:
import cv2
import numpy as np
# needed just for easier display in the notebook
from PIL import Image
import os
import time
from typing import Tuple, List, Final, Optional

## Model Configurations
The model parameters (NMS and confidence threshold and the input size to the model) as well as the paths to model weights and labels are define below as well. 

In [ ]:
# MODEL_WEIGHTS_PATH: Final[str] = '/home/cellareye/Development/yolov5/runs/train/cells_12_epochs/weights/cellpose_annots_12_epochs.onnx'
# CLASS_NAMES_PATH: Final[str] = '/home/cellareye/Development/yolov5/runs/train/cells_12_epochs/weights/cells.names'

MODEL_WEIGHTS_PATH: Final[str] = '//home/cellareye/Development/yolov5/runs/train/microscope-images-batch-1-091922-not-reviewed_20_epochs/weights/microscope-images-batch-1-091922-not-reviewed_20_epochs.onnx'
CLASS_NAMES_PATH: Final[str] = '/home/cellareye/Development/yolov5/runs/train/microscope-images-batch-1-091922-not-reviewed_20_epochs/weights/cells.names'
DEFAULT_DETECTION_CONFIDENCE: Final[float] = 0.4
DEFAULT_NMS_THRESHOLD: Final[float] = 0.3
INPUT_IMAGE_SIZE: Final[Tuple[int, int]] = (640, 640)  # width x height

## Utilities Functions

In [ ]:
# very efficient batch IoU calculation
# needed for combining detection results from different image crops on 
# the overlapping parts; we use this function instead of 
# torchvision.ops.box_iou(bboxes1, bboxes2) to remove dependency on torch
def iou_batch(bboxes1: np.ndarray, bboxes2: np.ndarray) -> np.ndarray:
    """Given Nx4 and Mx4 ndarrays of bounding boxes, compute pairwise IoUs"""
    # expand dims to allow computing pairwise IoU via outerproducts (creates NxM below)
    bboxes1 = np.expand_dims(bboxes1, 1)  # Nx1x4
    bboxes2 = np.expand_dims(bboxes2, 0)  # 1xMx4
    # determine the (x, y) coordinates of the intersection rectangle
    inter_x1s = np.maximum(bboxes1[..., 0], bboxes2[..., 0])  # pairwise max NxM
    inter_y1s = np.maximum(bboxes1[..., 1], bboxes2[..., 1])  # pairwise max NxM
    inter_x2s = np.minimum(bboxes1[..., 2], bboxes2[..., 2])  # pairwise min NxM
    inter_y2s = np.minimum(bboxes1[..., 3], bboxes2[..., 3])  # pairwise min NxM
    inter_ws = np.maximum(0., inter_x2s - inter_x1s)  # pairwise width of intersection rectangle NxM
    inter_hs = np.maximum(0., inter_y2s - inter_y1s)  # pairwise height of intersection rectangle NxM
    inter_areas = inter_ws * inter_hs  # pairwise intersection area NxM
    union_areas = ((bboxes1[..., 2] - bboxes1[..., 0])
                   * (bboxes1[..., 3] - bboxes1[..., 1])
                   + (bboxes2[..., 2] - bboxes2[..., 0])
                   * (bboxes2[..., 3] - bboxes2[..., 1])
                   - inter_areas + 1e-30)  # pairwise union area NxM
    return inter_areas / union_areas     # pairwise intersection divided by union (iou) NxM

# a function to return the area of bounding box
def box_area(box: np.array) -> float:
    """
    Args:
        box (numpy array of size (4,) or (4, 1) or (1, 4) or a 4-tuple or a 4-elements list): The box. 
        
    Return the area.
    """
    return (box[3] - box[1]) * (box[2] - box[0])

# checks on the results
def show_detections(image, boxes, labels, scores, label_map):
    
    # colors for displaying bounding boxes
    COLORS = [(0, 0, 255), (0, 255, 0), (255, 0, 0),
              (255, 0, 255), (0, 255, 255), (255, 255, 0)]
    
    class_ids = list(label_map.keys())
    if len(image.shape) < 3:
        input_image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
    else:
        input_image = image.copy()
    
    H, W = image.shape[:2]
    print('The image sizes are (H, W) = %d, %d' %(H, W))
    # read the annotations file
    for i, box in enumerate(boxes):
        (xtl, ytl, xbr, ybr) = box
        text = label_map[labels[i]]
        if labels[i] not in class_ids:
            print('Incorrect label was found %s' %label)
            # use black for incorrect label
            color = (0, 0, 0)
        else:
            color = COLORS[labels[i] % len(COLORS)]
                
        cv2.rectangle(input_image, (xtl, ytl), (xbr, ybr), color, 1)
        cv2.putText(input_image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    return Image.fromarray(input_image[:, :, (2, 1, 0)])

## YOLOv5 Detector Class
Make sure to pass the paths to the weights and names files if the default values are not to be used. 

In [ ]:
class Yolov5ObjectDetector:
    def __init__(self,
                 weights_path: Optional[str] = MODEL_WEIGHTS_PATH,
                 names_path: Optional[str] = CLASS_NAMES_PATH,
                 model_input_size: Tuple[int, int] = INPUT_IMAGE_SIZE,
                 confidence: float = DEFAULT_DETECTION_CONFIDENCE,
                 nms_threshold: float = DEFAULT_NMS_THRESHOLD):

        self._net = None
        self._weights_path: str = str(weights_path)
        self._names_path: str = str(names_path)
        self._model_input_size: Tuple[int, int] = model_input_size
        self._confidence: float = confidence
        self._nms_threshold: float = nms_threshold

        print(f'[INFO]: Loading class names from {self._names_path} with confidence {self._confidence} and NMS '
              f'threshold {self._nms_threshold} ...')
        
        # read class names and create the label map
        with open(self._names_path, 'r') as f:  # if fails to read then blow with error
            class_names: List[str] = [cname.strip() for cname in f.readlines()]

        self._label_map: Dict[int, str] = {i: c for i, c in enumerate(class_names)}
        self._reverse_label_map: Dict[str, int] = {value: key for key, value in self._label_map.items()}

        print('[INFO]: Class names were successfully loaded')
        
        # loading the ONNX model
        try:
            self._net = cv2.dnn.readNetFromONNX(self._weights_path)

            print(f'[INFO]: Loading YOLOv5 ONNX weights from {self._weights_path}. Setting dnn to use CUDA.')
            try:
                self._net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
                self._net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA_FP16)
            except Exception:
                print('[ERROR]: Failed to set dnn to use CUDA! Installed OpenCV is not compiled with CUDA support!')

        except Exception as ex:
            print(f"[ERROE]: Failed to initialize YOLOv5 model with CUDA. Likely the paths to model ONNX weights "
                          f"{self._weights_path} is incorrect: {repr(ex)}.")

    def detect(self, image: np.ndarray) -> Tuple[np.array, np.array, np.array]:
        """
        The main function to detect the bounding box and masks for persons in the input image.
        """
        if self._net is None:
            print("[ERROR]: YOLOv5 CV model has not been initialized. Please initialize the class before detect().")
            return np.zeros((0, 4), dtype=int), np.zeros((0,), dtype=int), np.zeros((0,), dtype=float)

        start: float = time.time()

        image_height, image_width = image.shape[:2]
        
        # check if the aspect ratio of the input image is almost the same as the aspect ratio of 
        # the model input size
        # if not, then the input image will be resized without keeping its aspect ratio when it 
        # is passed to the model, and this may lead to inaccurate detection
        aspect_ratio_diff: float = (image_width * self._model_input_size[1]) / (image_height *  self._model_input_size[0]) - 1
        
        if np.abs(aspect_ratio_diff) > 0.1:
            print("[WARN]: The input image has a different aspect ratio: {}".format(image_width / image_height) +  
                  "than the model: {}! The results may not be accurate".format(self._model_input_size[0] / self._model_input_size[1]))
            
        
        # prepare input blob and perform inference the model expect the image in 3 channel format
        # also, not need for scaling as the range is expected to be in 0-255
        if len(image.shape) < 3:
            input_image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
        else:
            input_image = image
        input_blob = cv2.dnn.blobFromImage(input_image, scalefactor=1.0 / 255,
                                           size=self._model_input_size,
                                           mean=(0, 0, 0), swapRB=True, crop=False)
        # set the input to the model
        self._net.setInput(input_blob)
        # run the forward pass to get output of the output layers
        outputs: np.ndarray = self._net.forward(self._net.getUnconnectedOutLayersNames())
        # only one output layer, pick the first element in the tuple then remove the batch dimension
        outputs = outputs[0][0]
        boxes, labels, scores = self._post_process(outputs, (image_width, image_height))

        elap: float = time.time() - start
        print(f"YOLO V5 object detection took {elap:.4f} seconds")
       
        return boxes, labels, scores

    def set_confidence(self, confidence):
        self._confidence = confidence

    def set_nms_threshold(self, threshold):
        self._nms_threshold = threshold

    def _post_process(self, outputs: np.ndarray, org_image_shape: Tuple[int, int]) -> Tuple[np.array, np.array, np.array]:
        """Post process outputs, discarding unreliable detections & performing NMS"""
        # discard unreliable detections before NMS to reduce NMS computations
        outputs = outputs[np.where(outputs[:, 4] >= min(0.5, self._confidence))[0]]
        # class IDs
        labels: np.ndarray = np.argmax(outputs[:, 5:], axis=1)
        # scores
        scores: np.ndarray = np.max(outputs[:, 5:], axis=1)
        # center (x, y) and width and height of each box
        cx: np.array = outputs[:, 0]
        cy: np.array = outputs[:, 1]
        w: np.array = outputs[:, 2]
        h: np.array = outputs[:, 3]
        # convert the boxes from (cx, cy, w, h) format to (xtl, ytl, w, h) before running NMS
        boxes: np.ndarray = np.vstack([cx - w/2, cy - h/2, w, h]).T * \
            np.array([org_image_shape[0] / self._model_input_size[0], org_image_shape[1] / self._model_input_size[1]] * 2)

        # run NMS per class
        index_list_to_return: List[int] = []
        # we are interested in unique class IDs in the output, hence set(classes)
        unique_label_ids = list(set(labels))
        for label_id in unique_label_ids:
            class_indexes = np.where(labels == label_id)[0]
            valid_det_indexes = cv2.dnn.NMSBoxes(boxes[class_indexes], scores[class_indexes], self._confidence, self._nms_threshold)

            index_list_to_return += list(class_indexes[valid_det_indexes])

        # NMS removes repeated detections (may return empty ndarrays)
        boxes = boxes[index_list_to_return, :]
        scores = scores[index_list_to_return]
        labels = labels[index_list_to_return]

        # convert from xtl, ytl, w, h) to (xtl, ytl, xbr, ybr) for each box
        boxes[:, 2] += boxes[:, 0]
        boxes[:, 3] += boxes[:, 1]
        # convert the coordinates to int
        boxes = boxes.astype(int)

        return boxes, labels, scores
    
    def detect_by_cropping(self, image: np.ndarray, crop_corners: List[List[int]], 
                           nms_threshold_for_combining_crop_results: float=0.1) \
    -> Tuple[np.array, np.array, np.array]:
        
        """
        A function to apply the model on a high resolution image. If the
        image is high resolution with many objects, after resizing the image
        to match the models input sizes, the objects may become too small for
        reliable detection. This function evaluate an image by first cropping
        the image into smaller and potentially overlapping sub-images (as
        specified by cropCorners), running the detector on each sub-image and
        then combining the detections over multiple overlapping sub-images
        by applying NMS
        Args:
            image (numpy array): Input image; it should be an OpenCV np.uint8 numpy 
            array either in BGR format or Grayscale (output of cv2.imread).
        crop_corners (list of 4-tuples (x1, y1, x2, y2)): Each element of
            this list specifies a cropped sub-image of the input image with
            top-left corner (x1, y1) and bottom-right corner (x2, y2). All
            the coordinates should be with respect to input image sizes.
            The input image is divided into len(crop_corners) sub-images
            before running the model on each.
        nms_threshold_for_combining_crop_results (float): NMS threshold to be used
            for combining detections of cropped sub-images over the overlapping areas. 
        Returns:
            numpy array of bounding boxes for detected objects; each bounding box
                is represented as (x1, y1, x2, y2) where (x1, y1) and (x2, y2)
                are the coordinates of the top-left and bottom-right corners
                of the bounding box, respectively.
            numpy array of labels for detected objects; the label index to class name
                mapping is specified by self.labelMap.
            numpy of detection scores for detected objects.
        """

        start = time.time()
        if len(crop_corners) == 0:
            print('[ERROR]: No crop corners are provided for running YOLOv5 model ' + 
                  'on sub-images. Returning no detections')
            return np.zeros((0, 4), dtype=int), np.zeros((0,), dtype=int), np.zeros((0,), dtype=float)


        H, W = image.shape[:2]

        # check if all the crop sub-images are of the same size,
        # if not, make them equal size for batch processing
        crop_widths: List[int] = [min(c[2], W) - max(c[0], 0) for c in crop_corners]
        crop_heights: List[int] = [min(c[3], H) - max(c[1], 0) for c in crop_corners]

        if min(crop_widths) <= 0 or min(crop_heights) <= 0:
            print('[ERROR]: Incorrect corners are provided for running YOLOv5 model ' + 
                  'on sub-images. Returning no detections')
            return np.zeros((0, 4), dtype=int), np.zeros((0,), dtype=int), np.zeros((0,), dtype=float)

        crop_width: int = max(crop_widths)
        crop_height: int = max(crop_heights)
            
        
        # combine the results, filter them based on the score,
        # and update the coordinates of the bounding boxes
        # for applying NMS later
        results: Dict = {'scores': [], 'boxes': [], 'labels': []}

        # a list to keep track of cropped sub-images with at least one object detection
        crop_ids_with_detection: List[int] = []

        for crop_id, corners in enumerate(crop_corners):
            (x1c, y1c, x2c, y2c) = corners
            # enlarge the crop if necessary to make all the same size
            x2c = x1c + crop_width
            y2c = y1c + crop_height

            # crop the image and run the model
            cropped_image = image[y1c:y2c, x1c:x2c]
            boxes, labels, scores = self.detect(cropped_image)

            # combine the detection results
            if len(scores) == 0:
                continue
                
            # find the bounding boxes close to the boundaries of the cropped image
            # these boxes most probably are truncated (because they are close to the boundary)
            # for a properly designed image crops, the overlapping section (in x or y dimensions)
            # between two adjacent crops is larger than than the largest object (in each dimension)
            # hence an object can only cross one boundary of an overlapping part and will
            # definitely lie completely in another cropped image
            # modify the score for these detected boxes (assign the minimum score of self._confidence)
            # to give them lower priority during NMS when the results in the overlapping parts
            # of cropped images are combined  

            scores[(boxes[:, 0] < 4) | (boxes[:, 1] < 4) | (boxes[:, 2] > crop_width - 4) |
                   (boxes[:, 3] > crop_height - 4)] = self._confidence


            crop_ids_with_detection.append(crop_id)
            results['scores'].append(scores)
            results['boxes'].append(boxes + np.array([x1c, y1c, x1c, y1c], dtype=int))
            results['labels'].append(labels)


        # no object detected, return
        if len(crop_ids_with_detection) == 0:
            return np.zeros((0, 4), dtype=int), np.zeros((0,), dtype=int), np.zeros((0,), dtype=float)

        # list to contain the detections
        boxes, labels, scores = [], [], []

        # now compare the detection results of one crop with the detections in the
        # rest of the image to identify objects that are uniquely detected in the
        # crop and should be kept
        # this part is needed to pick one object in the overlapping crop areas when
        # detected by the detector in multiple crops
        # note we need to compare the detections in one crop only with overlapping
        # crops; but here we are doing it for all for simplicity of implementation
        # TODO: improve it in the future by passing the indexes of the neighboring crops

        # to decide which object to keep when detected in multiple crops, we compute
        # the IoU (for objects of the same class) between the crop under consideration and the rest
        # then we keep objects in the crop under consideration that have
        # IoU <= nms_threshold_for_combining_crop_results with objects detected in the rest
        # we also keep objects with IoU > nms_threshold_for_combining_crop_results, if the detection
        # score for the object in the crop is higher than the objects in the other crops
        for idx, crop_id in enumerate(crop_ids_with_detection):

            crop_labels: np.array = results['labels'][idx]
            crop_scores: np.array = results['scores'][idx]
            crop_boxes: np.array = results['boxes'][idx]

            num_detections_in_rest = 0
            for i in range(len(crop_ids_with_detection)):
                if i != idx:
                    num_detections_in_rest += len(results['labels'][i])

            if num_detections_in_rest == 0:
                # keep all detections in this crop
                for i, label in enumerate(crop_labels):
                    boxes.append(crop_boxes[i])
                    labels.append(label)
                    scores.append(crop_scores[i])
                continue

            # labels  for detections in other cropped sub-images
            rest_labels: np.array = np.hstack([results['labels'][i] 
                                               for i in range(len(crop_ids_with_detection)) if i != idx])
            # scores for detections in other cropped sub-images
            rest_scores: np.array = np.hstack([results['scores'][i] 
                                               for i in range(len(crop_ids_with_detection)) if i != idx])
            # boxes for detections in other cropped sub-images
            rest_boxes: np.array = np.vstack([results['boxes'][i]
                                             for i in range(len(crop_ids_with_detection)) if i != idx])

            for label in set(list(crop_labels) + list(rest_labels)):
                
                # the indexes of detections of the same label in each crop and rest set
                crop_class_idxs: np.array = np.where(crop_labels == label)
                rest_class_idxs: np.array = np.where(rest_labels == label)

                if len(crop_class_idxs[0]) == 0:
                    continue

                if len(rest_class_idxs[0]) == 0:
                    # keep all detections for this label in this crop
                    idx_to_keep: np.array = np.array([i for i in range(len(crop_class_idxs[0]))], dtype=np.int)
                else:

                    # compute the IoU matrix, use torchvision implementation for efficiency
                    iou_matrix: np.array = iou_batch(crop_boxes[crop_class_idxs], rest_boxes[rest_class_idxs])

                    # keep the bounding boxes from the crop that have
                    # 1) IoU <= nms_threshold_for_combining_crop_results with the boxes in the rest 
                    #    of the crops, or
                    # 2) IoU > nms_threshold_for_combining_crop_results and the detection score for 
                    #    the box in the crop is higher than the detection scores for the boxes in the rest

                    # IoU <= nms_threshold_for_combining_crop_results
                    idx_to_keep = np.where(np.max(iou_matrix, axis=1) <= nms_threshold_for_combining_crop_results)[0]

                for i in range(len(crop_class_idxs[0])):
                    # detection score of this object
                    crop_det_score = crop_scores[crop_class_idxs[0][i]]
                    if i in idx_to_keep:
                        boxes.append(crop_boxes[crop_class_idxs[0][i]])
                        labels.append(crop_labels[crop_class_idxs[0][i]])
                        scores.append(crop_det_score)
                    else:
                        # there is some boxes in the rest of the crops with IoU more then the threshold
                        # find the maximum detection scores among the objects with IoU more than the threshold
                        # also find the area of the bounding box for that detection (needed to break a tie in case
                        # scores are equal)
                        high_iou_idxs = np.where(iou_matrix[i] > nms_threshold_for_combining_crop_results)[0]
                        scores_to_check = [rest_scores[rest_class_idxs[0][j]] for j in high_iou_idxs]
                        max_index = np.argmax(scores_to_check)
                        rest_det_score = scores_to_check[max_index]
                        
                        # areas of the matching boxes
                        crop_box_area: int = box_area(crop_boxes[crop_class_idxs[0][i]])
                        rest_box_area: int = box_area(rest_boxes[rest_class_idxs[0][high_iou_idxs[max_index]]])

                        # in case of a tie, pick the box with a larger area
                        # a tie can happen specially when the objects are both near a common boundary (e.g., a side of
                        # the image) of the crops (we reduce the scores for both to the threshold score and they become
                        # equal)
                        if crop_det_score > rest_det_score or \
                                (crop_det_score == rest_det_score and crop_box_area >= rest_box_area):
                            # keep this object as it has the highest score among all
                            boxes.append(crop_boxes[crop_class_idxs[0][i]])
                            labels.append(crop_labels[crop_class_idxs[0][i]])
                            scores.append(crop_det_score)
                            if crop_det_score == rest_det_score:
                                # this is added to break the tie if both areas are equal
                                # so we will not add the same box twice when considering 
                                # in another crop
                                results['scores'][idx][crop_class_idxs[0][i]] += 1e-5

        # change the coordinates from numpy array for each box to a list
        boxes = np.array(boxes)
        labels = np.array(labels)
        scores = np.array(scores)

        elap = time.time() - start
        print('YOLOv5 object detection after cropping the image to ' +
              '{} sub-images took {:.4f} seconds in OpenCV'.format(len(crop_corners), elap))
        return boxes, labels, scores

## Inference
### Instantiate the detector class
Make sure the default paths to the weights and names files are correct. Otherwise, pass the path for them to the class. 

In [ ]:
detector = Yolov5ObjectDetector()

### Run on 640x640 images
Make sure the image that is passed to the model is 640x640 (the YOLOv5 model input size), or at least has the same aspect ratio (square) for accurate detections. 

In [ ]:
# we are reading an already processed image below that is stored as a np.uint8 image
# that is why we skip the required normalization steps
img = cv2.imread('/home/cellareye/Development/yolov5/data/cells/images/test/Snapshot_489_crp_3.jpg', 
                 cv2.IMREAD_UNCHANGED)

In [ ]:
boxes, labels, scores = detector.detect(img)

In [ ]:
show_detections(img, boxes, labels, scores, detector._label_map)

In [ ]:
Image.fromarray(img)

### Run YOLOv5 on original resolution 2000x1600 images
Since YOLOv5 is trained on 640 x 640 images, in order to run it on images in the original resolution, we need
to crop the main image to square sub-images and then pass each to YOLOv5 and then compine the results. 
Below, we are reading a microscope image in its original resolution and format. Hence, pre-processing is required. 

- We first normalize the image (map [0 - 2^13 -1] color-depth to [0-255]).
- Then resize it to 1000x600 resolution. This resized image has enought resolution for object detection as YOLOv5 is trained on CellPose detections on images scaled by 1/2.
- We run YOLOv5 on 4 overlapping sub-images covering the large image and combine the results.
- Rescale the detection back to the original image resolution.

In [ ]:
org_img = cv2.imread('/home/cellareye/Cellanome/Images/img/org_images/Snapshot_489.png', 
                 cv2.IMREAD_UNCHANGED).astype(float)
org_img = (255 * org_img / (2 ** 14 - 1)).astype(np.uint8)
img = cv2.resize(org_img, (1000, 800))

In [ ]:
crop_corners = [[0, 0, 640, 640],
                [0, 160, 640, 800],
                [360, 0, 1000, 640],
                [360, 160, 1000, 800]]
boxes, labels, scores = detector.detect_by_cropping(img, crop_corners)
# scale the detections back to original image resolution
boxes = 2 * boxes

In [ ]:
show_detections(org_img, boxes, labels, scores, detector._label_map)

### Run YOLOv5 on other image resolutions
Use the function below to find the crop corners for other image resolutions

In [ ]:
def get_crop_corners(image_width: int, image_height: int, 
                     overlap_in_x: int=80, overlap_in_y: int=80, 
                     yolo_input_size: int=640) -> List[List[int]]:
    # the step size for the starting point of each crop in x and y dimension
    crop_start_step_x: int = yolo_input_size - overlap_in_x
    crop_start_step_y: int = yolo_input_size - overlap_in_y
    crop_corners: List = []

    # overlapping crops
    for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
        for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
            # crop coordinates
            xc_tl = x_start
            yc_tl = y_start
            xc_br = x_start + yolo_input_size
            yc_br = y_start + yolo_input_size
            # make sure we always crop the image with the given size
            # if we get to the boundaries, extend the crop
            # size inside the image to always get the same size crop
            # this is not really needed, but help with capturing more
            # annotations toward the low/right parts of the image
            if xc_br > image_width:
                xc_br = image_width 
                xc_tl = xc_br - yolo_input_size
            if yc_br > image_height:
                yc_br = image_height
                yc_tl = yc_br - yolo_input_size
           
            crop_coords = [xc_tl, yc_tl, xc_br, yc_br]
            crop_corners.append(crop_coords)
    
    return crop_corners


In [ ]:
org_img = cv2.imread('/home/cellareye/Downloads/scan_1/breadboard/1_1_1_1_1_027000_046400_-00085_BF.png', 
                     cv2.IMREAD_UNCHANGED)
print(org_img.shape)

In [ ]:
# in the following, we first resize the image to 2880x2880 and then find the crop corners using the function 
# above
img = cv2.cvtColor(org_img, cv2.COLOR_RGB2GRAY)
img = cv2.resize(img, (2880, 2880), interpolation = cv2.INTER_AREA)
crop_corners = get_crop_corners(image_width=2880, image_height=2880, 
                                overlap_in_x=80, overlap_in_y=80, 
                                yolo_input_size=640) 

In [ ]:
len(crop_corners)

In [ ]:
boxes, labels, scores = detector.detect_by_cropping(img, crop_corners)
# scale the detections back to original image resolution
boxes = (4512 / 2880 * boxes).astype(int)

In [ ]:
show_detections(org_img, boxes, labels, scores, detector._label_map)